In [0]:
from pyspark.sql.functions import col, sum, count, coalesce, lit, round, when

po = spark.table(
    "workspace.pharma_silver.silver_purchase_orders"
)

shipments = spark.table(
    "workspace.pharma_silver.silver_shipments"
)

suppliers = spark.table(
    "workspace.pharma_silver.silver_suppliers"
)

In [0]:
shipment_summary = (
    shipments
    .groupBy(
        "po_id",
        "supplier_id"
    )
    .agg(
        sum("quantity_shipped").alias("total_quantity_shipped")
    )
)

In [0]:
df_supplier_gold = (
    po
    .join(
        shipment_summary,
        on=["po_id", "supplier_id"],
        how="left"
    )
    .withColumn(
        "total_quantity_shipped",
        coalesce(col("total_quantity_shipped"), lit(0))
    )
)

In [0]:
df_supplier_gold = (
    df_supplier_gold
    .join(
        suppliers.select(
            "supplier_id",
            "supplier_name"
        ),
        on="supplier_id",
        how="left"
    )
)

In [0]:
df_supplier_gold = (
    df_supplier_gold
    .groupBy(
        "supplier_id",
        "supplier_name"
    )
    .agg(
        count("po_id").alias("total_orders"),
        sum("quantity_ordered").alias("total_quantity_ordered"),
        sum("total_quantity_shipped").alias("total_quantity_shipped")
    )
)

In [0]:
df_supplier_gold = df_supplier_gold.withColumn(
    "fulfillment_rate",
    round(
        when(
            col("total_quantity_ordered") > 0,
            (
                col("total_quantity_shipped")
                / col("total_quantity_ordered")
            ) * 100
        ).otherwise(0),
        2
    )
)

In [0]:
df_supplier_gold = df_supplier_gold.withColumn(
    "fulfillment_gap",
    col("total_quantity_ordered")
    - col("total_quantity_shipped")
)

In [0]:
df_supplier_gold = df_supplier_gold.withColumn(
    "performance_status",
    when(
        col("fulfillment_rate") >= 90,
        "Excellent"
    )
    .when(
        col("fulfillment_rate") >= 75,
        "Good"
    )
    .otherwise(
        "Needs Improvement"
    )
)

In [0]:
df_supplier_gold = df_supplier_gold.select(
    "supplier_id",
    "supplier_name",
    "total_orders",
    "total_quantity_ordered",
    "total_quantity_shipped",
    "fulfillment_rate",
    "fulfillment_gap",
    "performance_status"
)

In [0]:
display(df_supplier_gold)

In [0]:
df_supplier_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.pharma_gold.gold_supplier_performance"
    )